## Current Assignment

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
from bs4 import BeautifulSoup
import pandas as pd
import re
import requests
import os
from dotenv import load_dotenv

In [ ]:
#Setting up chrome driver
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
try:
    #Using this specific url because it lists movies sorted by number of votes
    url = "https://www.imdb.com/search/title/?title_type=feature&sort=num_votes,desc"
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    #Click the "Load More" button 200 times because 50 movies are loaded each time (50*200 = 10000)
    for i in range(200):
        try:
            #Wait for the button to be clickable
            load_more_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "ipc-see-more__button")))
            #Click the button
            driver.execute_script("arguments[0].click();", load_more_button)
            print(f"Clicked 'Load More' {i+1} times")
            #Pause to allow new content to load
            time.sleep(3) 
        except Exception as e:
            print("No more buttons found or error:", e)
            break
    #Extract the final HTML content after all the clicks
    page = driver.page_source
    print("HTML saved")
finally:
    #Close browser
    driver.quit()

In [3]:
#Parse the HTML using BeautifulSoup
soup = BeautifulSoup(page, 'html.parser')
#Extract "ipc-title-link-wrapper", which holds the title links
movies = soup.find_all('a', class_='ipc-title-link-wrapper')

imdb_ids = []
for movie in movies:
    #Extract the ID from the link using regex
    imdb_ids.append(re.search(r'/title/(tt\d+)/', movie['href']).group(1))
    
imdb_ids[:5]

['tt0111161', 'tt0468569', 'tt1375666', 'tt0137523', 'tt0816692']

In [4]:
#Convert the data to a pandas dataframe for easier saving
df = pd.DataFrame(imdb_ids, columns=['imdb_id'])
df.head()

,imdb_id
0,tt0111161
1,tt0468569
2,tt1375666
3,tt0137523
4,tt0816692


In [7]:
#Save to a CSV file
df.to_csv('imdb_top_movies.csv', index=False)

### OMDB API Data Collection
Fetch detailed metadata for each movie using the OMDB API.

Use a set to track already-fetched IDs. This allows the script to be rerun safely (if the connection drops or the daily 1,000-request limit is reached) without having to begin from scratch again.

Iterate through the IMDB IDs, sending a request to OMDB for each one and appending the JSON response to our list.

The collected metadata is converted to a Pandas DataFrame and saved as a CSV file.

In [ ]:
#Load API key for OMDB
load_dotenv()
OMDB_API_KEY = os.getenv('OMDB_API_KEY')

In [3]:
# Load the full list of IMDB IDs we want data for
imdb_ids_df = pd.read_csv('imdb_top_movies.csv')
imdb_ids = imdb_ids_df['imdb_id'].tolist()

# Load whatever OMDB data we already have (same file we save to)
omdb_data_df = pd.read_csv('full_movie_data.csv') if os.path.exists('full_movie_data.csv') else pd.DataFrame()

# Track already-fetched IDs so reruns are safe
fetched_ids = set(omdb_data_df['imdbID']) if not omdb_data_df.empty else set()

print(f"Already have {len(fetched_ids)} movies")
print(f"Remaining to fetch: {len(set(imdb_ids) - fetched_ids)}")

Already have 1999 movies
Remaining to fetch: 7401


In [4]:
omdb_data = []
fetch_count = 0
fail_count = 0

# Run until "remaining" prints as 0. Requests can fail and there's a
# 1000/day limit, so this is safe to rerun — already-fetched IDs are skipped.
for imdb_id in imdb_ids:
    if imdb_id in fetched_ids:
        continue

    if fail_count >= 100:
        print("Reached 100 failed requests, stopping.")
        break

    try:
        response = requests.get(
            "http://www.omdbapi.com/",
            params={"i": imdb_id, "apikey": OMDB_API_KEY},
            timeout=10
        )
        data = response.json()

        # OMDB returns 200 even for bad IDs/limit hit — check the Response field
        if response.status_code == 200 and data.get("Response") == "True":
            omdb_data.append(data)
            fetched_ids.add(imdb_id)   # avoid duplicates within this run
            fetch_count += 1
        else:
            error = data.get("Error", f"status {response.status_code}")
            print(f"Failed for {imdb_id}: {error}")
            fail_count += 1
    except Exception as e:
        print(f"Error for {imdb_id}: {e}")
        fail_count += 1

print(f"Fetched this run: {fetch_count}")
print(f"Failed this run: {fail_count}")
print(f"Remaining: {len(set(imdb_ids) - fetched_ids)}")

Error for tt0094862: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Error for tt7068946: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Failed for tt0120461: Request limit reached!
Failed for tt1619029: Request limit reached!
Failed for tt3554046: Request limit reached!
Failed for tt2452386: Request limit reached!
Failed for tt22048412: Request limit reached!
Failed for tt0107211: Request limit reached!
Failed for tt0463034: Request limit reached!
Failed for tt7014006: Request limit reached!
Failed for tt2125435: Request limit reached!
Failed for tt10059518: Request limit reached!
Failed for tt0075029: Request limit reached!
Failed for tt0048728: Request limit reached!
Failed for tt14114802: Request limit reached!
Failed for tt6265828: Request limit reached!
Failed for tt2649554: Request limit reached!
Failed

In [5]:
# Combine newly fetched rows with existing data
new_df = pd.DataFrame(omdb_data)
combined_df = pd.concat([omdb_data_df, new_df], ignore_index=True)

# Drop any accidental duplicates, keeping the first occurrence
if 'imdbID' in combined_df.columns:
    combined_df = combined_df.drop_duplicates(subset='imdbID', keep='first')

# Save back to the SAME file we load from, so the next run resumes correctly
combined_df.to_csv('full_movie_data.csv', index=False)
print(f"Saved {len(combined_df)} total movies")

omdb_data_df = combined_df  # update in-memory copy in case you rerun the fetch cell

Saved 2999 total movies
